# 03 — PyTorch baseline with CleanRL-style PPO

Wraps the legacy `NetForgeRLEnv` (PettingZoo) as a single-agent Gym env on the controlled agent, then trains stable-baselines3 PPO. Other agents act randomly.

In [ ]:
!pip install -q 'git+https://github.com/reforcemind/NetForge_RL' stable-baselines3 gymnasium

In [ ]:
import gymnasium as gym
import numpy as np
from netforge_rl.baselines import RandomPolicy
from netforge_rl.environment.parallel_env import NetForgeRLEnv

class SingleAgentWrapper(gym.Env):
    metadata = {'render_modes': []}
    def __init__(self, controlled='blue_dmz'):
        self.controlled = controlled
        self._env = NetForgeRLEnv({'scenario_type': 'ransomware', 'max_ticks': 100})
        self._env.reset(seed=0)
        self.action_space = self._env.action_space(controlled)
        self.observation_space = self._env.observation_space(controlled)['obs']
        self._others = RandomPolicy(seed=42)
    def reset(self, seed=None, options=None):
        obs, _ = self._env.reset(seed=seed or 0)
        return obs[self.controlled]['obs'], {}
    def step(self, action):
        acts = {a: self._others.act(self._env, a) for a in self._env.agents}
        acts[self.controlled] = np.asarray(action, dtype=np.int64)
        obs, r, term, trunc, _ = self._env.step(acts)
        return obs[self.controlled]['obs'], float(r.get(self.controlled, 0.0)), term.get(self.controlled, False), trunc.get(self.controlled, False), {}

from stable_baselines3 import PPO
model = PPO('MlpPolicy', SingleAgentWrapper(), verbose=1)
model.learn(total_timesteps=5_000)